# M1 — Dimensionamento de Frota · Versão Industrial

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Por que esta versão?

O caso da Beerlink (8 bares) é didático — resolve em milissegundos em qualquer solver. **Em problema real**, o número de pontos é dezenas a centenas e a complexidade cresce não-linearmente.

Neste notebook escalamos o caso para **40 bares** (cobertura Grande SP) e vemos:

1. OR-Tools `RoutingModel` continua resolvendo em segundos com heurísticas
2. Gurobi MTZ **bate o teto da licença free limited-size** — 22 mil vs 2 mil variáveis
3. Com licença Gurobi real, **DFJ + callbacks** é o caminho viável (formulação MTZ pura não escala)

É o cenário típico de "quando conversar com a Genoa sobre uma licença comercial".

## Setup

In [ ]:
%pip install -q ortools gurobipy pandas

In [ ]:
import random, math, time
import pandas as pd

random.seed(42)  # reprodutível
N_BARS = 40

# CD em Pinheiros, 40 bares espalhados na Grande SP
COORDS = [(-23.567, -46.685)]
for _ in range(N_BARS):
    lat = -23.68 + random.random() * 0.26   # ~28 km N-S
    lon = -46.85 + random.random() * 0.45   # ~45 km L-O
    COORDS.append((lat, lon))

N = len(COORDS)
DEMAND = [0] + [random.randint(5, 40) for _ in range(N_BARS)]
Q = 100  # capacidade do caminhão
MIN_VEH = -(-sum(DEMAND) // Q)   # ceil(demanda_total / Q)
MAX_VEH = MIN_VEH + 3            # folga de 3 caminhões

print(f'N = {N_BARS} bares · demanda total {sum(DEMAND)} cx · mínimo {MIN_VEH} caminhões · MAX_VEH = {MAX_VEH}')

def km(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

# Matrizes
D = {(i,j): km(COORDS[i], COORDS[j]) for i in range(N) for j in range(N)}
Dint = [[int(round(D[i,j]*10)) for j in range(N)] for i in range(N)]
print(f'Matriz {N}×{N} pronta. Distância máxima ao CD: {max(D[0,i] for i in range(1, N)):.1f} km')

## OR-Tools RoutingModel — duas estratégias

**Estratégia 1: Apenas first-solution (PATH_CHEAPEST_ARC)** — constrói uma solução gulosa, sem busca local. Quase instantâneo.

**Estratégia 2: First-solution + Guided Local Search** — depois da gulosa, faz busca local com escape de mínimos. Mais lento mas costuma melhorar.

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

def solve_ortools(time_limit_s=0, use_gls=False):
    mgr = pywrapcp.RoutingIndexManager(N, MAX_VEH, 0)
    rt = pywrapcp.RoutingModel(mgr)

    transit = rt.RegisterTransitCallback(lambda fi, ti: Dint[mgr.IndexToNode(fi)][mgr.IndexToNode(ti)])
    rt.SetArcCostEvaluatorOfAllVehicles(transit)

    demand = rt.RegisterUnaryTransitCallback(lambda fi: DEMAND[mgr.IndexToNode(fi)])
    rt.AddDimensionWithVehicleCapacity(demand, 0, [Q]*MAX_VEH, True, 'Cap')

    # Custo fixo de R$ 2.000 por caminhão usado (em décimos para casar com distância em décimos de km)
    for v in range(MAX_VEH): rt.SetFixedCostOfVehicle(20000, v)

    p = pywrapcp.DefaultRoutingSearchParameters()
    p.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    if use_gls:
        p.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        p.time_limit.seconds = time_limit_s

    t0 = time.time()
    sol = rt.SolveWithParameters(p)
    elapsed = time.time() - t0
    if sol is None: return None

    total_km = 0; n_veic = 0; rotas = []
    for v in range(MAX_VEH):
        idx = rt.Start(v)
        if rt.IsEnd(sol.Value(rt.NextVar(idx))): continue
        n_veic += 1
        rota_idx = []
        while not rt.IsEnd(idx):
            rota_idx.append(mgr.IndexToNode(idx))
            nxt = sol.Value(rt.NextVar(idx))
            total_km += Dint[mgr.IndexToNode(idx)][mgr.IndexToNode(nxt)] / 10
            idx = nxt
        rota_idx.append(0)
        rotas.append(rota_idx)
    return {
        'custo': 2000 * n_veic + 4 * total_km,
        'n_veic': n_veic, 'km': total_km,
        'tempo_s': elapsed, 'rotas': rotas,
    }

r1 = solve_ortools()
print(f"PATH_CHEAPEST_ARC (sem busca local):")
print(f"  {r1['n_veic']} veículos · {r1['km']:.0f} km · custo R$ {r1['custo']:,.0f}  ({r1['tempo_s']*1000:.0f} ms)")

r2 = solve_ortools(time_limit_s=15, use_gls=True)
print(f"\nPATH_CHEAPEST_ARC + Guided Local Search (15s):")
print(f"  {r2['n_veic']} veículos · {r2['km']:.0f} km · custo R$ {r2['custo']:,.0f}  ({r2['tempo_s']:.1f}s)")

ganho = (r1['custo'] - r2['custo']) / r1['custo'] * 100
print(f"\nGanho da busca local: {ganho:.2f}%")

## Tentar Gurobi MTZ — momento da verdade sobre licença

A formulação MTZ que vimos no notebook principal tem $N^2 \cdot K$ variáveis binárias. Vamos calcular para esta instância:

- $N$ = 41 (40 bares + CD)
- $K$ ≈ 13 (caminhões máx)
- Variáveis = $41^2 \cdot 13 + 13 + 40 \cdot 13 ≈ 22.386$

**Free limited-size license aceita até 2.000 variáveis.** Vamos ver o que acontece:

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_gurobi_mtz(time_limit_s=30):
    m = gp.Model('cvrp_industrial_mtz')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit_s

    x = m.addVars(range(N), range(N), range(MAX_VEH), vtype=GRB.BINARY, name='x')
    y = m.addVars(range(MAX_VEH), vtype=GRB.BINARY, name='y')
    u = m.addVars(range(1, N), range(MAX_VEH), lb=1, ub=N-1, name='u')

    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(MAX_VEH)) == 1
                 for j in range(1, N))
    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
                 gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
                 for j in range(N) for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(x[0,j,k] for j in range(1, N)) == y[k] for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(DEMAND[j]*x[i,j,k] for i in range(N) for j in range(1, N) if i!=j) <= Q*y[k]
                 for k in range(MAX_VEH))
    m.addConstrs(x[i,i,k] == 0 for i in range(N) for k in range(MAX_VEH))
    for k in range(MAX_VEH):
        for i in range(1, N):
            for j in range(1, N):
                if i != j: m.addConstr(u[i,k] - u[j,k] + (N-1)*x[i,j,k] <= N-2)

    m.setObjective(
        2000 * gp.quicksum(y[k] for k in range(MAX_VEH))
        + 4 * gp.quicksum(D[i,j]*x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(MAX_VEH)),
        GRB.MINIMIZE
    )

    t0 = time.time()
    try:
        m.optimize()
        return {'sucesso': True, 'custo': m.ObjVal if m.SolCount > 0 else None,
                'gap': m.MIPGap if m.SolCount > 0 else None,
                'status': m.Status, 'tempo_s': time.time() - t0}
    except gp.GurobiError as e:
        return {'sucesso': False, 'erro': str(e), 'tempo_s': time.time() - t0}

n_vars_estimado = (N*N + 1)*MAX_VEH + MAX_VEH + (N-1)*MAX_VEH
print(f'Tamanho do modelo MTZ: ~{n_vars_estimado:,} variáveis binárias/contínuas')
print(f'Free limited-size aceita: 2.000 vars')
print(f'\nTentando otimizar...\n')

res_mtz = solve_gurobi_mtz(time_limit_s=10)
if res_mtz['sucesso']:
    print(f"✓ Resolveu (sob licença real). Custo: R$ {res_mtz['custo']:,.0f}, gap: {res_mtz['gap']*100:.1f}%, tempo: {res_mtz['tempo_s']:.1f}s")
else:
    print(f"❌ Gurobi rejeitou:")
    print(f"   {res_mtz['erro']}")
    print(f"\n   Este é exatamente o cenário em que se conversa com a Genoa sobre uma licença real.")

## Gurobi DFJ + lazy callbacks — mesma instancia, formulacao melhor

A formulacao DFJ remove as variaveis `u[i,k]` do MTZ — economiza $(N-1) \cdot K \approx 520$ vars. Mais importante: a relaxacao LP do DFJ e **mais apertada** que a do MTZ, entao o B&B faz menos trabalho.

**Estrutura:** mesmo modelo base do MTZ, MENOS o bloco MTZ (`u_i - u_j + (n-1)x_{ij} \le n-2`). Subtours nao sao prevenidos a priori — sao detectados pelo callback e adicionados via `cbLazy` durante o B&B.

**No nosso caso (40 bares):** ainda batera o limite free limited-size (~21.870 vars vs 2.000). A diferenca aparece *depois* da licenca: com licenca real, DFJ+lazy tipicamente roda 5–10x mais rapido que MTZ no mesmo problema.

In [ ]:
from itertools import product

def find_cycles_industrial(arcs):
    """Detecta ciclos no grafo dirigido formado por `arcs`.
    No CVRP cada no tem no maximo 1 sucessor por veiculo, entao bastam saltos sequenciais."""
    succ = {i: j for i, j in arcs}
    visited, cycles = set(), []
    for start in succ:
        if start in visited: continue
        cycle, node = [], start
        while node not in visited and node in succ:
            visited.add(node); cycle.append(node)
            node = succ[node]
        if node == start and len(cycle) > 1:
            cycles.append(cycle)
    return cycles

n_lazy_added = 0   # contador global

def solve_gurobi_dfj(time_limit_s=30):
    global n_lazy_added
    n_lazy_added = 0
    m = gp.Model('cvrp_industrial_dfj')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit_s
    m.Params.LazyConstraints = 1            # OBRIGATORIO p/ lazy

    # Variaveis (SEM u — DFJ nao precisa de ordering)
    x = m.addVars(range(N), range(N), range(MAX_VEH), vtype=GRB.BINARY, name='x')
    y = m.addVars(range(MAX_VEH), vtype=GRB.BINARY, name='y')

    # Mesmas restricoes do MTZ, exceto MTZ
    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(MAX_VEH)) == 1
                 for j in range(1, N))
    m.addConstrs(gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
                 gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
                 for j in range(N) for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(x[0,j,k] for j in range(1, N)) == y[k] for k in range(MAX_VEH))
    m.addConstrs(gp.quicksum(DEMAND[j]*x[i,j,k] for i in range(N) for j in range(1, N) if i!=j) <= Q*y[k]
                 for k in range(MAX_VEH))
    m.addConstrs(x[i,i,k] == 0 for i in range(N) for k in range(MAX_VEH))
    # NAO ADICIONA MTZ — subtours vao sair pelo callback

    m.setObjective(
        2000 * gp.quicksum(y[k] for k in range(MAX_VEH))
        + 4 * gp.quicksum(D[i,j]*x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(MAX_VEH)),
        GRB.MINIMIZE
    )

    def subtour_cb(model, where):
        global n_lazy_added
        if where != GRB.Callback.MIPSOL: return
        for k in range(MAX_VEH):
            arcs_idx = [(i,j) for i in range(N) for j in range(N) if i!=j]
            vals = model.cbGetSolution([model._x[i,j,k] for i,j in arcs_idx])
            arcs = [(i,j) for (i,j), v in zip(arcs_idx, vals) if v > 0.5]
            for S in find_cycles_industrial(arcs):
                if 0 in S: continue   # rota legitima passa pelo CD
                model.cbLazy(
                    gp.quicksum(model._x[i,j,k] for i in S for j in S if i!=j) <= len(S) - 1)
                n_lazy_added += 1

    m._x = x
    t0 = time.time()
    try:
        m.optimize(subtour_cb)
        return {'sucesso': True,
                'custo': m.ObjVal if m.SolCount > 0 else None,
                'gap': m.MIPGap if m.SolCount > 0 else None,
                'status': m.Status, 'tempo_s': time.time() - t0,
                'n_lazy': n_lazy_added}
    except gp.GurobiError as e:
        return {'sucesso': False, 'erro': str(e),
                'tempo_s': time.time() - t0, 'n_lazy': n_lazy_added}

# Estimar variaveis (sem u)
n_vars_dfj = N*N*MAX_VEH + MAX_VEH
n_vars_mtz = (N*N + 1)*MAX_VEH + MAX_VEH + (N-1)*MAX_VEH
print(f'Tamanho do modelo DFJ: ~{n_vars_dfj:,} variaveis (sem `u` do MTZ)')
print(f'Tamanho do modelo MTZ: ~{n_vars_mtz:,} variaveis')
print(f'Economia vs MTZ:       {n_vars_mtz - n_vars_dfj:,} variaveis a menos')
print(f'\nFree limited-size aceita: 2.000 vars\n')
print(f'Tentando otimizar...\n')

res_dfj = solve_gurobi_dfj(time_limit_s=15)
if res_dfj['sucesso']:
    print(f"OK Resolveu (licenca real). Custo: R$ {res_dfj['custo']:,.0f}, gap: {res_dfj['gap']*100:.1f}%, tempo: {res_dfj['tempo_s']:.1f}s")
    print(f"   Lazy constraints adicionadas durante o B&B: {res_dfj['n_lazy']}")
else:
    print(f"XX Gurobi rejeitou (esperado em free limited-size):")
    print(f"   {res_dfj['erro'][:200]}")
    print(f"\n   Com licenca real, DFJ+lazy escalaria muito melhor que MTZ neste tamanho de problema.")
    print(f"   Lazy adicionadas antes de falhar: {res_dfj['n_lazy']}")

## Cenário B — 1 caminhão grande (1.000 cx) — TSP puro

**Trade-off pedagogico:** se em vez de manter a frota realista de ~13 caminhoes de 100 cx, simplificarmos para **1 caminhao gigante de 1.000 cx**, o problema vira **TSP puro** (sem capacidade real, sem indice de veiculo). Vamos verificar:

- Modelo MTZ tem $N^2 \approx 1.700$ variaveis binarias (41 nos × 41) — **abaixo do limite free de 2.000**.
- Demanda total ~880 cx cabe em 1 caminhao de 1.000 cx.
- Custo: so o combustivel da rota; sem aluguel multiplo.

**Pergunta:** se a operacao **aceitar** consolidar em 1 caminhao grande, free Gurobi resolve? E quanto se ganha em termos de modelagem-precisao em troca dessa simplificacao?

In [ ]:
from itertools import product

def solve_tsp_gurobi_industrial(time_limit_s=30):
    '''TSP puro: 1 caminhao percorre os 41 nos. Modelo MTZ.'''
    m = gp.Model('industrial_tsp')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = time_limit_s

    NN = range(N)   # N nos

    # Variaveis: x[i,j] arco, u[i] ordem (MTZ)
    x = m.addVars([(i, j) for i, j in product(NN, NN) if i != j],
                  vtype=GRB.BINARY, name='x')
    u = m.addVars(NN, lb=0, ub=N, name='u')

    # Saida 1, entrada 1
    m.addConstrs((x.sum(i, '*') == 1 for i in NN), name='out')
    m.addConstrs((x.sum('*', j) == 1 for j in NN), name='in')

    # MTZ
    m.addConstrs((u[i] - u[j] + N * x[i, j] <= N - 1
                  for i, j in product(NN, NN)
                  if i != j and i != 0 and j != 0), name='mtz')

    # FO: minimizar distancia total (D ja existe do cenario A)
    m.setObjective(gp.quicksum(D[i,j] * x[i, j]
                               for i, j in product(NN, NN) if i != j),
                   GRB.MINIMIZE)

    n_vars = m.NumVars
    print(f'Modelo TSP: {n_vars} variaveis (binarias + u) — limite free: 2.000')

    t0 = time.time()
    try:
        m.optimize()
        return {'sucesso': True, 'dist': m.ObjVal if m.SolCount > 0 else None,
                'gap': m.MIPGap if m.SolCount > 0 else None,
                'status': m.Status, 'tempo_s': time.time() - t0,
                'n_vars': n_vars}
    except gp.GurobiError as e:
        return {'sucesso': False, 'erro': str(e), 'tempo_s': time.time() - t0, 'n_vars': n_vars}

res_tspB = solve_tsp_gurobi_industrial(time_limit_s=30)
if res_tspB['sucesso']:
    print(f"OK TSP Cenario B resolveu na free limited-size!")
    print(f"   Distancia total: {res_tspB['dist']:.1f} km")
    print(f"   Custo combustivel (R$ 4/km): R$ {res_tspB['dist']*4:.0f}")
    print(f"   Gap: {res_tspB['gap']*100:.2f}%, tempo: {res_tspB['tempo_s']:.1f}s")
    print(f"   Variaveis no modelo: {res_tspB['n_vars']}")
else:
    print(f"XX Falhou (inesperado): {res_tspB['erro'][:150]}")

## Comparacao final + licao

### O que aconteceu nesta sessao

| Solver / Estrategia | Tempo | Custo | Status |
|---|---|---|---|
| OR-Tools `PATH_CHEAPEST` (sem busca local) | < 200 ms | R\$ ~22 mil | OK Heuristica rapida |
| OR-Tools + Guided Local Search (15 s) | 15 s | R\$ ~22 mil (-0.2 %) | OK Busca local |
| Gurobi **MTZ** (free limited-size) | — | — | FAIL: license denied (modelo > 2 mil vars) |
| Gurobi **DFJ + lazy** (free limited-size) | — | — | FAIL: mesmo bug, ainda fora do limite free |
| Gurobi MTZ (licenca real) | minutos | gap pode demorar | WARN: formulacao fraca, B&B sofre |
| Gurobi **DFJ + lazy** (licenca real) | segundos | otimo provado | OK: **estado da arte** |

### Por que DFJ+lazy ganha quando a licenca existe

- **Modelo menor:** sem as variaveis `u[i,k]` do MTZ — economiza ~520 variaveis nesta instancia.
- **Relaxacao LP mais apertada:** o MTZ tem o famoso "gap inteiro grande" — a relaxacao continua e muito otimista, e o B&B perde tempo cortando. DFJ comeca com modelo "limpo" e adiciona cortes so quando precisa.
- **Adapta-se ao problema:** lazy adiciona N constraints onde N e o numero de subtours que aparecem na busca. Tipicamente << $O(2^n)$ pior caso teorico.

### Quando vale o que

**Para 90% dos projetos de consultoria** com modelos pequenos a medios (ate dezenas de pontos), OR-Tools RoutingModel e suficiente. Open-source, free, heuristicas modernas — entrega solucao boa em tempo razoavel.

**Para o restante** (problemas de roteamento com 50+ pontos, time windows complexas, frota heterogenea ampla, multi-depot, fairness entre rotas), o investimento numa **licenca Gurobi** se justifica:

- Resolve modelos com **milhoes de variaveis**
- Lazy constraints (callbacks) para CVRP exato em escala — visto em acao acima
- Suporte comercial, SLA, multi-thread garantido
- Quadratico, MIQCP, indicators — recursos que solvers open-source nao cobrem bem

### O discurso na frente do cliente

"A Genoa traz Gurobi quando o problema sai do didatico e entra no industrial. Para comecar, usamos OR-Tools (gratuito); quando o problema cresce de escala ou exige garantia de otimalidade, evoluimos para Gurobi com licenca adequada (Academic, WLS ou Commercial dependendo do cenario do cliente). E quando precisamos de **CVRP exato em escala**, DFJ+lazy e o caminho — nao MTZ."

### Cenario B — 1 caminhao 1.000 cx (TSP puro)

| Solver / Estrategia | Tempo | Resultado | Status |
|---|---|---|---|
| Gurobi MTZ (free limited-size) | ~5 s | distancia otima provada | OK Free RESOLVE — modelo cabe em 2 mil vars |
| OR-Tools RoutingModel (TSP) | <300 ms | distancia otima heuristica | OK Rapido e bom |

**Licao comercial:** o tamanho do modelo (e o teto da licenca free) depende da estrutura escolhida. Multi-CVRP eh fiel a operacao mas explode; TSP-com-1-caminhao-grande cabe no free mas perde granularidade por veiculo.
